# Chemotherapy case study: 5-FU and capecitabine

This notebook applies the compartment-driven Gaussian-process framework to
two **aggregated cohort-mean** concentration--time curves digitized from the
published clinical-pharmacology study of capecitabine and its metabolite
5-fluorouracil (5-FU) in patients with normal liver function versus liver
metastasis-driven hepatic dysfunction (Twelves et al., 1999, *Clinical
Cancer Research*).

For each drug there are two cohort trajectories:
- `blank`  — hepatic dysfunction cohort (open / unfilled markers in the source figure)
- `filled` — normal liver function cohort (filled markers in the source figure)

Each trajectory comes as a sequence of mean plasma concentrations $\bar Y_j$
with per-timepoint standard errors $SE_j$. There are no individual patient
trajectories available, so the inferential target is the *cohort-mean
concentration-time curve* and the population-mean AUC and $C_{\max}$.

The clinical question is whether liver dysfunction shifts AUC. The published
study reports only point estimates of AUC and a $p$-value; the
compartment-driven GP turns the cohort-mean curves into posterior
distributions over AUC, letting us read off a posterior probability of
$AUC^{\text{dysfunction}} > AUC^{\text{normal}}$ directly.

## Setup

In [ ]:
import os, sys, warnings, time
warnings.filterwarnings('ignore')
from pathlib import Path
CODE_DIR = Path(os.environ.get('PKGP_CODE_DIR', Path.cwd())).expanduser().resolve()
if not (CODE_DIR / 'scenarios.py').is_file():
    raise RuntimeError('Run from code/ or set PKGP_CODE_DIR to its absolute path.')
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
OUTPUT_DIR = Path(os.environ.get('PKGP_OUTPUT_DIR', CODE_DIR)).expanduser().resolve()
os.environ['PKGP_OUTPUT_DIR'] = str(OUTPUT_DIR)
DATA_DIR = CODE_DIR / 'data'
RES_DIR = OUTPUT_DIR / 'results'
FIG_DIR = OUTPUT_DIR / 'figures'
RES_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import methods
from models import one_cpt_oral

plt.rcParams.update({
    'font.size': 10,
    'axes.titlesize': 11,
    'axes.labelsize': 10,
    'legend.fontsize': 9,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'lines.linewidth': 1.2,
    'figure.dpi': 100,
})
os.makedirs(FIG_DIR, exist_ok=True); os.makedirs(RES_DIR, exist_ok=True)

## Load aggregated cohort-mean curves

In [ ]:
df_5fu = pd.read_csv(str(DATA_DIR / '5fu_cleaned.csv'))
df_cap = pd.read_csv(str(DATA_DIR / 'capecitabine_cleaned.csv'))

print('5-FU:')
print(df_5fu.to_string(index=False))
print()
print('Capecitabine:')
print(df_cap.to_string(index=False))

## Population-typical (popPK) mean functions

We use one-compartment oral absorption mean functions with population-typical
rate parameters chosen to match the qualitative shape of the cohort-mean
curve (rise and fall, peak time). The Gaussian-process discrepancy absorbs
any mean-function misfit, so these popPK values are starting points rather
than precise fits.

In [ ]:
# Population-typical 1-cpt oral absorption parameters for the qualitative
# shape of each drug's plasma concentration profile (parent compound for
# capecitabine, downstream metabolite for 5-FU). The dose normalization is
# absorbed into the volume since we only need the shape; the GP discrepancy
# rescales the mean to match the data.
POP_5FU = dict(ka=1.0, ke=0.3, V=10.0, dose=1.0)
POP_CAP = dict(ka=2.0, ke=0.5, V=2.0,  dose=10.0)

def mean_func_for(pop):
    def mf(t):
        return one_cpt_oral(np.asarray(t, dtype=float),
                             pop['ka'], pop['ke'], pop['V'], pop['dose'])
    return mf

## Fit the compartment-driven GP to each of the four trajectories

In [ ]:
KERNEL = 'matern52'  # smooth kernel for the smooth aggregated cohort-mean curves
# Prior medians for the kernel marginal SD, chosen so that sigma_f is of the
# order of the peak concentration scale of each drug (well above the reported
# per-timepoint SE), so the GP posterior at each observation is governed by
# (data, SE) rather than shrinking back toward the popPK prior. The log-normal
# prior (log-SD PRIOR_LOG_SD) replaces the old hard sigma_f floor.
SIGMA_F_SCALE_5FU = 0.3   # peak ~0.2, max SE ~0.074
SIGMA_F_SCALE_CAP = 4.0   # peak ~5.6, max SE ~1.8
# Fixed GP length scale of 1.5 hr, roughly the inter-observation spacing.
ELL_FIXED = 1.5     # length scale held fixed (hr)
PRIOR_LOG_SD = 0.2

def fit_trajectory(df, drug_label, trajectory_label, pop, sigma_f_scale,
                    a, b, n_paths=2000, seed=42):
    sub = df[df.trajectory == trajectory_label].sort_values('x').reset_index(drop=True)
    # Drop the artificial (0,0,0) anchor row used by the digitization, which
    # has SE=0 and would be treated as a noise-free observation (collapses
    # the posterior to exactly y=0 at t=0, which the popPK mean already does).
    sub = sub[sub['se'] > 0].reset_index(drop=True)
    t_obs = sub['x'].to_numpy(dtype=float)
    y_obs = sub['y'].to_numpy(dtype=float)
    se_obs = sub['se'].to_numpy(dtype=float)
    rng = np.random.default_rng(seed)
    mf = mean_func_for(pop)
    res = methods.method_mech_gp(t_obs, y_obs, mf, a, b,
                                  n_paths=n_paths, rng=rng,
                                  kernel=KERNEL,
                                  sigma_f_scale=sigma_f_scale,
                                  sigma_eps_fixed=se_obs,
                                  ell_fixed=ELL_FIXED,
                                  prior_log_sd=PRIOR_LOG_SD)
    res['_t_obs'] = t_obs; res['_y_obs'] = y_obs; res['_se_obs'] = se_obs
    res['_drug'] = drug_label
    res['_trajectory'] = trajectory_label
    res['_mean_func'] = mf
    res['_window'] = (a, b)
    return res

t0_5fu, t_end_5fu = 0.0, 7.0
t0_cap, t_end_cap = 0.0, 6.0

fits = {
    ('5-FU',         'blank'):  fit_trajectory(df_5fu, '5-FU',         'blank',  POP_5FU, SIGMA_F_SCALE_5FU, t0_5fu, t_end_5fu, seed=1),
    ('5-FU',         'filled'): fit_trajectory(df_5fu, '5-FU',         'filled', POP_5FU, SIGMA_F_SCALE_5FU, t0_5fu, t_end_5fu, seed=2),
    ('Capecitabine', 'blank'):  fit_trajectory(df_cap, 'Capecitabine', 'blank',  POP_CAP, SIGMA_F_SCALE_CAP, t0_cap, t_end_cap, seed=3),
    ('Capecitabine', 'filled'): fit_trajectory(df_cap, 'Capecitabine', 'filled', POP_CAP, SIGMA_F_SCALE_CAP, t0_cap, t_end_cap, seed=4),
}
for k, v in fits.items():
    sf, el, _ = v['_hyper']
    print(f'{k[0]:14s} {k[1]:6s}: AUC = {v["auc"]:.3f} ({v["auc_lo"]:.3f}, {v["auc_hi"]:.3f})  '
          f'(sigma_f = {sf:.3f}, ell = {el:.3f})')

## Figure: posterior fits for each trajectory

Each row of the figure is one drug. The left panel shows the **dysfunction**
cohort and the right panel the **normal** cohort, with observed cohort-mean
concentrations and per-timepoint SE error bars, the posterior mean curve,
and the 95% pointwise band.

In [ ]:
def plot_drug(drug_label, sigma_f_min, out_path, ylim=None):
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
    for ax, traj in zip(axes, ['blank', 'filled']):
        r = fits[(drug_label, traj)]
        grid = r['_paths_grid']
        mu = r['_mu_grid']
        sd = np.sqrt(np.maximum(r['_K_grid_diag'], 0.0))
        mech = r['_mean_func'](grid)
        ax.fill_between(grid, mu - 1.96 * sd, mu + 1.96 * sd,
                        color='#d62728', alpha=0.18, label='posterior 95% band')
        ax.plot(grid, mu,   color='#d62728', lw=1.6, label='posterior mean')
        ax.errorbar(r['_t_obs'], r['_y_obs'], yerr=r['_se_obs'],
                    fmt='o', color='black', markersize=4, capsize=2,
                    label='cohort mean ± SE')
        label = 'hepatic dysfunction' if traj == 'blank' else 'normal liver function'
        ax.set_title(f'{drug_label}: {label}')
        ax.set_xlabel('time (hr)')
        ax.set_ylabel('plasma concentration')
        ax.grid(alpha=0.3, lw=0.5)
        if ylim is not None:
            ax.set_ylim(ylim)
    axes[0].legend(loc='upper right', fontsize=8)
    fig.tight_layout()
    fig.savefig(out_path, bbox_inches='tight')
    plt.show()
    plt.close(fig)

plot_drug('5-FU',         SIGMA_F_SCALE_5FU,
           os.path.join(FIG_DIR, 'fig_5fu_patients.pdf'),
           ylim=(-0.10, 0.40))
plot_drug('Capecitabine', SIGMA_F_SCALE_CAP,
           os.path.join(FIG_DIR, 'fig_capecitabine_patients.pdf'),
           ylim=(-2.5, 9.0))

## Figure: posterior AUC distributions, dysfunction vs. normal

For each drug, the posterior AUC distribution is plotted for the dysfunction
and normal trajectories. The probability that dysfunction has higher AUC than
normal is computed by paired sampling from the two posterior distributions
(treated as independent since the two cohorts in the source study were
disjoint patient groups).

In [ ]:
def plot_auc_comparison(out_path):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.0))
    for ax, drug in zip(axes, ['5-FU', 'Capecitabine']):
        b = fits[(drug, 'blank')]['auc_samples']
        f = fits[(drug, 'filled')]['auc_samples']
        # Histogram of each posterior
        for samp, color, label in [(b, '#9467bd', 'hepatic dysfunction'),
                                    (f, '#2ca02c', 'normal liver function')]:
            ax.hist(samp, bins=40, density=True, color=color, alpha=0.5,
                    edgecolor='white', label=label)
        # Posterior probability of difference > 0
        n = min(len(b), len(f))
        diff = b[:n] - f[:n]
        p = float((diff > 0).mean())
        ax.set_title(f'{drug}: P(AUC dysfunction > AUC normal) = {p:.3f}')
        ax.set_xlabel('AUC')
        ax.set_yticks([])
        ax.grid(axis='x', alpha=0.3, lw=0.5)
        ax.legend(loc='upper right', fontsize=8)
    fig.tight_layout()
    fig.savefig(out_path, bbox_inches='tight')
    plt.show()
    plt.close(fig)

plot_auc_comparison(os.path.join(FIG_DIR, 'fig_chemo_auc_comparison.pdf'))

## Posterior summary table

In [ ]:
rows = []
for (drug, traj), r in fits.items():
    rows.append({
        'drug': drug,
        'cohort': 'dysfunction' if traj == 'blank' else 'normal',
        'auc_mean': float(r['auc']),
        'auc_sd':   float(r['auc_sd']),
        'auc_lo':   float(r['auc_lo']),
        'auc_hi':   float(r['auc_hi']),
        'cmax_median': float(np.median(r['cmax_samples'])),
        'cmax_lo':     float(r['cmax_lo']),
        'cmax_hi':     float(r['cmax_hi']),
    })
summary = pd.DataFrame(rows)
summary.to_csv(os.path.join(RES_DIR, 'chemo_posterior_summary.csv'), index=False)
summary.round(3)

## Posterior probability of AUC shift

In [ ]:
print('Posterior Pr(AUC_dysfunction > AUC_normal):')
shift_rows = []
for drug in ['5-FU', 'Capecitabine']:
    b = fits[(drug, 'blank')]['auc_samples']
    f = fits[(drug, 'filled')]['auc_samples']
    n = min(len(b), len(f))
    diff = b[:n] - f[:n]
    p = float((diff > 0).mean())
    mean_shift = float(np.mean(diff))
    sd_shift   = float(np.std(diff))
    lo, hi     = np.quantile(diff, [0.025, 0.975])
    shift_rows.append({'drug': drug, 'prob_positive': p, 'mean_shift': mean_shift, 'shift_lo': float(lo), 'shift_hi': float(hi)})
    print(f'  {drug:14s}: Pr(>0) = {p:.3f}   '
          f'mean shift = {mean_shift:+.3f}   '
          f'95% CI [{lo:+.3f}, {hi:+.3f}]')
pd.DataFrame(shift_rows).to_csv(RES_DIR / 'chemo_shift_summary.csv', index=False)
